In [27]:
import json
import math
import time
from pathlib import Path
from datetime import datetime, timezone

import requests

In [28]:
import json
from pathlib import Path

inputs = {
    "words": json.loads(Path("sample/words.json").read_text(encoding="utf-8")),
    "sentences": json.loads(Path("sample/sentences.json").read_text(encoding="utf-8")),
    "documents": json.loads(Path("sample/documents.json").read_text(encoding="utf-8")),
}

In [41]:
MODEL_LABEL = "Qwen3-Embedding-0.6B.i1-Q2_K.gguf"

LLAMA_SERVER_URL = "http://localhost:8111"
EMBEDDINGS_URL = f"{LLAMA_SERVER_URL}/v1/embeddings"
OUTPUT_PATH = Path(f"embeddings/{MODEL_LABEL.replace('/', '__')}.json")

In [ ]:
def clean_text(text: str) -> str:
    return "\n".join(line.strip() for line in str(text).strip().splitlines() if line.strip())


def vector_norm(vec: list[float]) -> float:
    return math.sqrt(sum(x * x for x in vec))


def load_inputs(sample_dir: str | Path = "sample") -> dict[str, list[str]]:
    sample_dir = Path(sample_dir)

    inputs = {
        "words": json.loads((sample_dir / "words.json").read_text(encoding="utf-8")),
        "sentences": json.loads((sample_dir / "sentences.json").read_text(encoding="utf-8")),
        "documents": json.loads((sample_dir / "documents.json").read_text(encoding="utf-8")),
    }

    cleaned_inputs: dict[str, list[str]] = {}

    for group_name, texts in inputs.items():
        if not isinstance(texts, list):
            raise TypeError(f"{group_name}.json must contain a JSON array")

        cleaned = [clean_text(text) for text in texts]
        cleaned = [text for text in cleaned if text]

        cleaned_inputs[group_name] = cleaned

    return cleaned_inputs


def embed_texts(texts: list[str], batch_size: int = 8) -> list[list[float]]:
    all_embeddings: list[list[float]] = []

    for start in range(0, len(texts), batch_size):
        batch = texts[start:start + batch_size]

        payload = {
            "model": MODEL_LABEL,
            "input": batch,
            "encoding_format": "float",
        }

        response = requests.post(
            EMBEDDINGS_URL,
            headers={
                "Content-Type": "application/json",
                "Authorization": "Bearer no-key",
            },
            json=payload,
            timeout=300,
        )

        if not response.ok:
            raise RuntimeError(
                f"Embedding request failed: HTTP {response.status_code}\n"
                f"{response.text}"
            )

        data = response.json()

        rows = sorted(data["data"], key=lambda x: x["index"])
        embeddings = [row["embedding"] for row in rows]

        if len(embeddings) != len(batch):
            raise RuntimeError(
                f"Embedding count mismatch: input={len(batch)}, output={len(embeddings)}"
            )

        all_embeddings.extend(embeddings)

        print(
            f"  batch {start // batch_size + 1}: "
            f"{start + len(batch)}/{len(texts)} done"
        )

    return all_embeddings


inputs = load_inputs("sample")

print("Loaded inputs:")
for group_name, texts in inputs.items():
    print(f"- {group_name}: {len(texts)}")


started_at = time.time()

result = {
    "model_label": MODEL_LABEL,
    "server_url": LLAMA_SERVER_URL,
    "embeddings_url": EMBEDDINGS_URL,
    "created_at": datetime.now(timezone.utc).isoformat(),
    "groups": {},
}

for group_name, texts in inputs.items():
    print(f"\nEmbedding group: {group_name}, count={len(texts)}")

    embeddings = embed_texts(texts)

    items = []
    for index, (text, embedding) in enumerate(zip(texts, embeddings)):
        items.append({
            "index": index,
            "text": text,
            "dimension": len(embedding),
            "norm": vector_norm(embedding),
            "embedding": embedding,
        })

    result["groups"][group_name] = {
        "count": len(items),
        "items": items,
    }

result["elapsed_seconds"] = round(time.time() - started_at, 3)

OUTPUT_PATH.write_text(
    json.dumps(result, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print(f"Elapsed: {result['elapsed_seconds']} seconds")

for group_name, group in result["groups"].items():
    dims = [item["dimension"] for item in group["items"]]
    norms = [item["norm"] for item in group["items"]]

    print(
        group_name,
        "count=", group["count"],
        "dim=", sorted(set(dims)),
        "norm_min=", min(norms),
        "norm_max=", max(norms),
    )